# w9_fullcorpus_build.ipynb — 全库(23,373)测试 · 阶段1 数据

User (2026-07-23):突破 2020,**完全不过滤**(min_length=0, min_count=1)。

**关键修正(用户抓出的设计错误)**:第一版打算端到端重跑 build.py,那会
**重嵌已有 2,020 游戏的 73,289,517 句**——纯浪费。正确做法:

- 只嵌 **21,353 个缺口游戏**(catalog 23,107 − 已嵌 2,020)
- **MERGE** 进现有池:老 2,020 **排在前面**(gidx 0..2019 不变)
  → wiki_eval 的 814 个 gidx、tag_labels、所有既有资产**全部继续有效**
- 最终全库 = 2,020(含 266 个 kaggle-only) + 21,353 = **23,373 游戏**

原始文本已在桶 `build_new_gamedata/`,直接 stage。稀疏游戏(总句数≤2·W=32)
阶段2 走纯-CE fallback。AUTO-STOPS 关闭。


In [ ]:
# constants
import os
REPO = "/workspace/stable-query-latent"
URL  = "https://github.com/Nice9Tian/stable-query-latent.git"
BUCKET = "s3://0wov6gbp6j"
ENDPOINT = "https://s3api-us-ks-2.runpod.io"
RAW_PREFIX = f"{BUCKET}/stable-query-latent/game_review_data/build_new_gamedata"
DATA_DIR = "/workspace/fullcorpus_data"      # gap-only build workdir
DATA_SRC = "/workspace/fusion_cache_w9"      # existing pool + assets live here
MIN_LENGTH, MIN_COUNT = 0, 1                 # NO FILTER (user decree)
VIEW_W = 16                                  # sparse threshold = 2*W = 32
os.makedirs(DATA_DIR, exist_ok=True)
print(f"gap-only full-corpus build -> {DATA_DIR} "
      f"(no filter; sparse if total sentences <= {2*VIEW_W})")

In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git fetch origin main && git reset --hard origin/main && git rev-parse --short HEAD
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
print("repo synced")

In [ ]:
# Stage raw review text + compute the GAP (catalog - already embedded).
# The existing pool's meta is the authority on what is already embedded.
import subprocess, os
from pathlib import Path
import numpy as np
MEND = "Steam Games Metadata and Player Reviews (2020\u20132024"   # en-dash
for src, dst in [(f"{RAW_PREFIX}/{MEND}/", f"{DATA_DIR}/{MEND}/"),
                 (f"{RAW_PREFIX}/kaggle_steam_reviews_prepared/",
                  f"{DATA_DIR}/kaggle_steam_reviews_prepared/")]:
    print(f"sync {src} ...", flush=True)
    subprocess.run(["aws", "s3", "sync", src, dst, "--endpoint-url", ENDPOINT,
                    "--only-show-errors"], check=True)
subprocess.run(["aws", "s3", "cp", f"{RAW_PREFIX}/games.json",
                f"{DATA_DIR}/games.json", "--endpoint-url", ENDPOINT,
                "--only-show-errors"], check=True)

GR = Path(DATA_DIR) / MEND / "Game Reviews"
catalog = {p.stem.split("_")[0]: p for p in GR.glob("*.csv")}
meta = np.load(Path(DATA_SRC) / "full_pool_meta.npz", allow_pickle=True)
kept_names = [str(x) for x in meta["game_names"]]
kept = {n.split("_")[0] for n in kept_names}
gap = sorted(set(catalog) - kept)
N_OLD_SENTS = int(meta["review_offsets"][-1])
N_OLD_REVS  = int(meta["game_review_offsets"][-1])
NG_OLD      = len(kept_names)
print(f"catalog {len(catalog):,} | already embedded {NG_OLD:,} "
      f"({N_OLD_REVS:,} reviews / {N_OLD_SENTS:,} sentences)")
print(f"GAP to embed: {len(gap):,} games   (re-embedding the old pool would "
      f"waste {N_OLD_SENTS:,} sentence-embeds -- we skip it)")

# hard-link ONLY the gap CSVs into a dedicated dir so every downstream stage
# processes exactly the new games.
NEW_REV = Path(DATA_DIR) / "gap_reviews"
NEW_REV.mkdir(exist_ok=True)
made = 0
for a in gap:
    d = NEW_REV / catalog[a].name
    if not d.exists():
        try: os.link(catalog[a], d)
        except OSError: __import__("shutil").copyfile(catalog[a], d)
        made += 1
print(f"gap review CSVs staged: {made:,} new / {len(gap):,} total -> {NEW_REV}")

In [ ]:
# GAP metadata (NO FILTER) + scale report. Only the 21k new games.
import subprocess, sys, glob, json
from pathlib import Path
import numpy as np
cmd = [sys.executable, "game_review_data/build_metadata.py",
       "--reviews-dir", str(Path(DATA_DIR) / "gap_reviews"),
       "--games-json", f"{DATA_DIR}/games.json",
       "--output-dir", f"{DATA_DIR}/metadata",
       "--min-length", str(MIN_LENGTH), "--min-count", str(MIN_COUNT),
       "--workers", "16"]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
files = sorted(glob.glob(f"{DATA_DIR}/metadata/*.json"))
rc = []
for f in files:
    d = json.loads(Path(f).read_text())
    rc.append(max(0, len(d) - 3))          # cleaned_3 always prepends 3 descs
rc = np.array(rc)
print(f"\n=== GAP SCALE (no filter) ===\ngap games kept: {len(files):,}")
print(f"gap reviews: {int(rc.sum()):,}  median/game {int(np.median(rc))}")
for t in (1, 2, 5, 10, 50):
    print(f"  games with >= {t:3d} reviews: {int((rc>=t).sum()):,}")

In [ ]:
# SPLIT the gap games (SaT). chunk-budget guards the half-precision OOM.
import subprocess, sys, glob, json
from pathlib import Path
import numpy as np
cmd = [sys.executable, "game_review_data/split_data.py",
       "--input-dir", f"{DATA_DIR}/metadata",
       "--output-dir", f"{DATA_DIR}/sentences",
       "--chunk-budget", "2000000"]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
sc = []
for f in sorted(glob.glob(f"{DATA_DIR}/sentences/*.json")):
    d = json.loads(Path(f).read_text())
    n = sum(len(v) for v in d.values()) if isinstance(d, dict) else len(d)
    sc.append(n)
sc = np.array(sc)
print(f"\n=== GAP SENTENCE SCALE ===\ngap sentences: {int(sc.sum()):,}")
print(f"  vs existing pool 73,289,517 -> merged total "
      f"{73_289_517 + int(sc.sum()):,}")
print(f"sparse games (<= {2*VIEW_W} sents -> plain-CE arm): "
      f"{int((sc<=2*VIEW_W).sum()):,} / {len(sc):,} "
      f"({(sc<=2*VIEW_W).mean():.1%})")

In [ ]:
# TEXT-H5 + EMBED the gap games only -> gap_embedding_h5.h5.
# NOTE: build.py cannot drive this -- with --skip-source1/2 its review_dirs is
# empty and it exits; call the same library functions directly on the gap dirs.
import sys
from pathlib import Path
sys.path.insert(0, f"{REPO}/game_review_data")
from h5_corpus import build_text_h5
from embedding_data import embed_data
import h5py

GAP_TEXT = f"{DATA_DIR}/gap_text_h5.h5"
GAP_EMB  = f"{DATA_DIR}/gap_embedding_h5.h5"
build_text_h5(
    sentences_dir=Path(f"{DATA_DIR}/sentences"),
    games_json=Path(f"{DATA_DIR}/games.json"),
    output_h5=Path(GAP_TEXT),
    reviews_dirs=[Path(DATA_DIR) / "gap_reviews"],
    label_min_length=MIN_LENGTH,
)
print("gap text_h5 built", flush=True)
embed_data(input_h5=GAP_TEXT, output_h5=GAP_EMB, backend="local",
           dtype="float16")
with h5py.File(GAP_EMB, "r") as h:
    print(f"gap h5: games {h['game_names'].shape[0]:,} "
          f"reviews {int(h['game_review_offsets'][-1]):,} "
          f"sentences {h['vectors'].shape[0]:,}")

In [ ]:
# MERGE: existing pool (OLD FIRST -> gidx 0..2019 preserved) + gap games.
# Streaming; rebases review_offsets by N_old_sents and game_review_offsets by
# N_old_revs. Writes a NEW pool + meta, atomically, next to the old one.
import os, time, shutil
from pathlib import Path
import numpy as np, h5py

OLD_V = Path(DATA_SRC) / "full_pool_fp16.npy"
OLD_M = Path(DATA_SRC) / "full_pool_meta.npz"
GAP_H5 = f"{DATA_DIR}/gap_embedding_h5.h5"
NEW_V = Path(DATA_SRC) / "full_pool_fc_fp16.npy"      # fc = full corpus
NEW_M = Path(DATA_SRC) / "full_pool_fc_meta.npz"

om = np.load(OLD_M, allow_pickle=True)
o_gro, o_ro = om["game_review_offsets"], om["review_offsets"]
o_names = [str(x) for x in om["game_names"]]
N_old_s, N_old_r, NG_old = int(o_ro[-1]), int(o_gro[-1]), len(o_names)
ov = np.load(OLD_V, mmap_mode="r")
assert ov.shape[0] == N_old_s, (ov.shape, N_old_s)

with h5py.File(GAP_H5, "r") as h:
    g_gro, g_ro = h["game_review_offsets"][:], h["review_offsets"][:]
    g_names = [x.decode() if isinstance(x, bytes) else str(x)
               for x in h["game_names"][:]]
    N_new_s, N_new_r, NG_new = int(g_ro[-1]), int(g_gro[-1]), len(g_names)
    assert h["vectors"].shape[0] == N_new_s
    print(f"merging: old {NG_old:,}g/{N_old_s:,}s + new {NG_new:,}g/{N_new_s:,}s"
          f" -> {NG_old+NG_new:,}g/{N_old_s+N_new_s:,}s", flush=True)
    # meta first (cheap, and proves the arithmetic before the 150GB copy)
    m_ro  = np.concatenate([o_ro,  g_ro[1:]  + N_old_s])
    m_gro = np.concatenate([o_gro, g_gro[1:] + N_old_r])
    m_names = np.array(o_names + g_names, dtype=object)
    assert m_ro[-1] == N_old_s + N_new_s and m_gro[-1] == N_old_r + N_new_r
    assert len(m_names) == NG_old + NG_new
    assert m_names[:NG_old] == o_names, "old games must stay first (gidx!)"
    np.savez(NEW_M, game_review_offsets=m_gro, review_offsets=m_ro,
             game_names=m_names)
    print("meta merged + arithmetic verified", flush=True)
    # vectors: stream old then new
    tmp = NEW_V.with_suffix(".npy.tmp")
    out = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16,
                                    shape=(N_old_s + N_new_s, 1024))
    t0, CH = time.time(), 1 << 20
    for i in range(0, N_old_s, CH):
        out[i:i+CH] = ov[i:i+CH]
    print(f"old vectors copied ({(time.time()-t0)/60:.1f} min)", flush=True)
    for i in range(0, N_new_s, CH):
        out[N_old_s+i:N_old_s+i+min(CH, N_new_s-i)] = h["vectors"][i:i+CH]
    out.flush(); del out
    os.replace(tmp, NEW_V)
print(f"merged pool -> {NEW_V}")

In [ ]:
# VERIFY the merge, then publish + upload.
from pathlib import Path
import numpy as np, h5py
m = np.load(Path(DATA_SRC) / "full_pool_fc_meta.npz", allow_pickle=True)
v = np.load(Path(DATA_SRC) / "full_pool_fc_fp16.npy", mmap_mode="r")
gro, ro, names = m["game_review_offsets"], m["review_offsets"], m["game_names"]
assert v.shape[0] == int(ro[-1]) and len(names) == len(gro) - 1
ov = np.load(Path(DATA_SRC) / "full_pool_fp16.npy", mmap_mode="r")
# old block must be byte-identical, new block must be non-zero
for i in (0, len(ov)//2, len(ov)-1):
    assert np.array_equal(v[i], ov[i]), f"old block changed at {i}"
for i in (len(ov), len(ov) + (len(v)-len(ov))//2, len(v)-1):
    assert float(np.abs(v[i]).sum()) > 0, f"new block zero at {i}"
print(f"VERIFIED: {len(names):,} games, {v.shape[0]:,} sentences; "
      f"old block intact, new block populated")
Path(DATA_SRC, "full_pool_fc_READY").write_text(f"fullcorpus {v.shape[0]}")
import subprocess
for f in ("full_pool_fc_fp16.npy", "full_pool_fc_meta.npz"):
    subprocess.run(["aws", "s3", "cp", str(Path(DATA_SRC)/f),
                    f"{BUCKET}/fusion_cache_w9/{f}", "--endpoint-url",
                    ENDPOINT, "--only-show-errors"], check=True)
subprocess.run(["aws", "s3", "cp", f"{DATA_DIR}/gap_embedding_h5.h5",
                f"{BUCKET}/fullcorpus/gap_embedding_h5.h5",
                "--endpoint-url", ENDPOINT, "--only-show-errors"], check=True)
print("uploaded merged pool + gap h5")

In [ ]:
# AUTO-STOP the pod.
import subprocess, os
print("stage-1 (gap embed + merge) complete")
subprocess.run(["runpodctl", "stop", "pod",
                os.environ.get("RUNPOD_POD_ID", "")], check=False)